## Imports

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm

from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc

/proj/berzelius-2025-315/users/x_dilwi/conda_envs/h5env_clean_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

CSV_PATH = Path("/proj/berzelius-2025-315/users/x_dilwi/bomi2_image_level_with_survival.csv")

RUNS = {
    "MAE200_DINOe99": Path("/proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch200/extracted_embeddings_epoch100"),
    "MAE300_DINOe99": Path("/proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch300/extracted_embeddings_epoch100"),
    "MAE400_DINOe99": Path("/proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch400/extracted_embeddings_epoch100"),
    "MAE500_DINOe99": Path("/proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch500/extracted_embeddings_epoch100"),
}

EMB_FILE = "embeddings_epoch=99.npy"
NAME_FILE = "names_epoch=99.npy"


def decode_name(x):
    """Convert numpy bytes/object/string into clean Python string."""
    if isinstance(x, bytes):
        return x.decode("utf-8")
    if isinstance(x, np.bytes_):
        return x.astype(str)
    return str(x)


def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


# -----------------------------------------------------------------------------
# 1) Check CSV
# -----------------------------------------------------------------------------
print_header("CHECKING CSV")

print(f"CSV exists: {CSV_PATH.exists()}")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

print(f"CSV shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum().sort_values(ascending=False))

print(f"\nTotal duplicated rows: {df.duplicated().sum()}")

# Optional: inspect a likely name column
possible_name_cols = [c for c in df.columns if "name" in c.lower() or "core" in c.lower() or "image" in c.lower()]
print(f"\nPossible name/id columns in CSV: {possible_name_cols}")

print("\nFirst 5 rows:")
print(df.head())


# -----------------------------------------------------------------------------
# 2) Check each run folder and embedding files
# -----------------------------------------------------------------------------
summary = []

for run_name, run_dir in RUNS.items():
    print_header(f"CHECKING RUN: {run_name}")

    emb_path = run_dir / EMB_FILE
    name_path = run_dir / NAME_FILE

    print(f"Run dir exists: {run_dir.exists()} -> {run_dir}")
    print(f"Embedding file exists: {emb_path.exists()} -> {emb_path}")
    print(f"Name file exists: {name_path.exists()} -> {name_path}")

    if not run_dir.exists():
        print("Skipping: run directory missing.")
        continue
    if not emb_path.exists() or not name_path.exists():
        print("Skipping: embedding or name file missing.")
        continue

    # Load arrays
    embeddings = np.load(emb_path, allow_pickle=True)
    names = np.load(name_path, allow_pickle=True)

    names = np.array([decode_name(x) for x in names])

    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Embeddings dtype: {embeddings.dtype}")
    print(f"Names shape: {names.shape}")
    print(f"Names dtype: {names.dtype}")

    # Size checks
    if embeddings.ndim != 2:
        print(f"WARNING: embeddings should usually be 2D [N, D], got shape {embeddings.shape}")

    if len(embeddings) != len(names):
        print(f"ERROR: number of embeddings ({len(embeddings)}) != number of names ({len(names)})")
    else:
        print(f"OK: embeddings and names have matching length = {len(names)}")

    # NA / inf checks in embeddings
    if np.issubdtype(embeddings.dtype, np.number):
        n_nan = np.isnan(embeddings).sum()
        n_inf = np.isinf(embeddings).sum()
        print(f"NaN count in embeddings: {n_nan}")
        print(f"Inf count in embeddings: {n_inf}")

        if embeddings.size > 0:
            print(f"Embedding min: {np.nanmin(embeddings)}")
            print(f"Embedding max: {np.nanmax(embeddings)}")
            print(f"Embedding mean: {np.nanmean(embeddings)}")
            print(f"Embedding std: {np.nanstd(embeddings)}")
    else:
        print("WARNING: embeddings are not numeric dtype.")

    # Name checks
    name_series = pd.Series(names)
    print(f"Missing/empty names: {(name_series.isna().sum()) + (name_series.astype(str).str.strip() == '').sum()}")
    print(f"Duplicate names: {name_series.duplicated().sum()}")
    print("\nFirst 10 names:")
    print(names[:10])

    summary.append({
        "run": run_name,
        "run_dir_exists": run_dir.exists(),
        "emb_file_exists": emb_path.exists(),
        "name_file_exists": name_path.exists(),
        "n_embeddings": len(embeddings),
        "embedding_dim": embeddings.shape[1] if embeddings.ndim == 2 else None,
        "n_names": len(names),
        "duplicate_names": int(name_series.duplicated().sum()),
        "nan_in_embeddings": int(np.isnan(embeddings).sum()) if np.issubdtype(embeddings.dtype, np.number) else None,
        "inf_in_embeddings": int(np.isinf(embeddings).sum()) if np.issubdtype(embeddings.dtype, np.number) else None,
    })

# -----------------------------------------------------------------------------
# 3) Summary table
# -----------------------------------------------------------------------------
print_header("SUMMARY")
summary_df = pd.DataFrame(summary)
print(summary_df)

## Load clinical CSV + Basic cleaning

In [3]:
clinical = pd.read_csv(CSV_PATH)

# expected columns: patient_id, time, event, image_name (from your snippet)
required = ["patient_id", "time", "event"]
missing = [c for c in required if c not in clinical.columns]
if missing:
    raise ValueError(f"CSV missing columns: {missing}. Available: {clinical.columns.tolist()}")

# Make sure types are right
clinical["patient_id"] = clinical["patient_id"].astype(str)
clinical["time"] = pd.to_numeric(clinical["time"], errors="coerce")
clinical["event"] = pd.to_numeric(clinical["event"], errors="coerce").astype(int)

clinical = clinical.dropna(subset=["time", "event", "patient_id"]).copy()
clinical["event"] = clinical["event"].astype(bool)

print("Clinical rows:", len(clinical))
print("Unique patients:", clinical["patient_id"].nunique())
print("Columns:", clinical.columns.tolist())
clinical.head()

Clinical rows: 542
Unique patients: 298
Columns: ['image_path', 'image_name', 'sample_name', 'row', 'col', 'patient_id_raw', 'patient_id', 'region_area_mm2', 'sample_region', 'cohort', 'Histology', 'Age', 'Sex', 'Smoking', 'Stage (7th ed.)', 'Performance status (WHO)', 'Follow-up (days)', 'Dead/Alive', 'time', 'event']


,image_path,image_name,sample_name,row,col,patient_id_raw,patient_id,region_area_mm2,sample_region,cohort,Histology,Age,Sex,Smoking,Stage (7th ed.),Performance status (WHO),Follow-up (days),Dead/Alive,time,event
0,/histodata/TIL/Lung_BOMI2_TIL/output TIFF/BOMI...,"BOMI2_TIL_1_Core[1,1,A]_[5091,35249]_component...","BOMI2_TIL_1_[1,A]",1,A,Lung # 479,479,0.107100,T,Lung,Squamous cell carcinoma,71.0,Female,Current smoker,Ib,1.0,4294.0,Alive,4294.0,False
1,/histodata/TIL/Lung_BOMI2_TIL/output TIFF/BOMI...,"BOMI2_TIL_1_Core[1,1,C]_[8663,35138]_component...","BOMI2_TIL_1_[1,C]",1,C,Lung # 561,561,0.606719,T,Lung,Adenocarcinoma,62.0,Female,Current smoker,Ia,1.0,3977.0,Alive,3977.0,False
2,/histodata/TIL/Lung_BOMI2_TIL/output TIFF/BOMI...,"BOMI2_TIL_1_Core[1,1,D]_[10473,35233]_componen...","BOMI2_TIL_1_[1,D]",1,D,Lung # 766,766,0.278944,T,Lung,Adenocarcinoma,59.0,Male,Ex-smoker,Ib,1.0,3227.0,Alive,3227.0,False
3,/histodata/TIL/Lung_BOMI2_TIL/output TIFF/BOMI...,"BOMI2_TIL_1_Core[1,1,E]_[12330,35106]_componen...","BOMI2_TIL_1_[1,E]",1,E,Lung # 743,743,0.422967,T,Lung,Adenocarcinoma,64.0,Female,Never-smoker,Ia,0.0,3266.0,Alive,3266.0,False
4,/histodata/TIL/Lung_BOMI2_TIL/output TIFF/BOMI...,"BOMI2_TIL_1_Core[1,1,G]_[15997,34963]_componen...","BOMI2_TIL_1_[1,G]",1,G,Lung # 779,779,0.389901,T,Lung,Squamous cell carcinoma,75.0,Female,Current smoker,Ia,0.0,2666.0,Dead,2666.0,True


## Helper: Load embeddings+names

In [4]:
def load_run_embeddings(run_dir: Path):
    E = np.load(run_dir / EMB_FILE, allow_pickle=True)
    N = np.load(run_dir / NAME_FILE, allow_pickle=True)
    N = np.array([str(x) for x in N], dtype=object)
    return E, N

## Helper: Map embedding names into clinical rows

In [5]:
def build_embedding_meta(names: np.ndarray) -> pd.DataFrame:
    """
    Create a dataframe with at least an 'image_name' or 'core_key' we can merge on.
    names in your DINO extraction look like: something from embedding_dataset:
        roi_names.append('_'.join(grpn)) where slide=core and roi='core'
    In your embedding_dataset.py, df['core'] is everything up to '_component_data'
    and slide = core. Then roi_names = f"{slide}_core".
    So names likely look like: "<..._component_data>_core"
    """
    df = pd.DataFrame({"name_raw": names})

    # common pattern from embedding_dataset: "<core_stem>_core"
    # where core_stem is "BOMI2_TIL_..._component_data"
    df["core_stem"] = df["name_raw"].str.replace(r"_core$", "", regex=True)

    # create possible join keys:
    # 1) exact image_name in CSV includes ".tif", but core_stem in your dataset may not.
    # try to build something comparable:
    df["image_name_guess"] = df["core_stem"].astype(str)

    # if your CSV has "image_name", we can try matching by prefix (without extension)
    return df


def attach_clinical(df_emb_meta: pd.DataFrame, clinical_df: pd.DataFrame) -> pd.DataFrame:
    """
    Tries to merge embedding meta with clinical using best available key.
    """
    c = clinical_df.copy()

    # if CSV has image_name, build a stem without extension for matching
    if "image_name" in c.columns:
        c["image_name"] = c["image_name"].astype(str)
        c["image_stem"] = c["image_name"].str.replace(r"\.tif$", "", regex=True)
    else:
        c["image_stem"] = None

    # Strategy A: merge on stem match
    if "image_name" in clinical_df.columns:
        merged = df_emb_meta.merge(
            c,
            left_on="core_stem",
            right_on="image_stem",
            how="left",
            suffixes=("", "_clin"),
        )
        matched = merged["patient_id"].notna().sum()
        if matched > 0:
            print(f"Matched via core_stem ↔ image_stem: {matched}/{len(merged)}")
            return merged

    # Strategy B: if not matched, try regex extraction to get "BOMI2_TIL_..._component_data.tif" from raw
    # (Adjust if your naming differs)
    pat = re.compile(r"(BOMI2_TIL_\d+_Core\[[^\]]+\]_\[\d+,\d+\]_[^/]*?component_data)\.?", re.IGNORECASE)
    df2 = df_emb_meta.copy()
    df2["image_stem_extracted"] = df2["name_raw"].apply(lambda s: pat.search(s).group(1) if pat.search(s) else None)

    if "image_name" in clinical_df.columns:
        merged = df2.merge(
            c,
            left_on="image_stem_extracted",
            right_on="image_stem",
            how="left",
            suffixes=("", "_clin"),
        )
        matched = merged["patient_id"].notna().sum()
        print(f"Matched via regex extracted stem ↔ image_stem: {matched}/{len(merged)}")
        return merged

    # Strategy C: fallback merge on patient only (rarely possible without losing cores)
    raise RuntimeError("Could not merge embeddings to clinical. Need a shared key (e.g. image_name).")

## Aggregate core embeddings into patient embeddings

In [6]:
def patient_pooling(E: np.ndarray, meta: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Mean-pool core embeddings per patient_id.
    Returns:
      X_pat: (n_patients, d)
      df_pat: columns include patient_id, time, event
    """
    meta_ok = meta.dropna(subset=["patient_id", "time", "event"]).copy()
    meta_ok["patient_id"] = meta_ok["patient_id"].astype(str)

    groups = []
    rows = []

    # each row of E corresponds to meta row index
    for pid, sub in meta_ok.groupby("patient_id"):
        idx = sub.index.to_numpy()
        vec = E[idx].mean(axis=0)
        groups.append(vec)

        # choose first row's survival info (should be identical per patient)
        r = sub.iloc[0][["patient_id", "time", "event"]].copy()
        rows.append(r)

    X = np.stack(groups, axis=0)
    df = pd.DataFrame(rows).reset_index(drop=True)

    return X, df

## CV Evaluation C-index + time-dependent AUC

In [16]:
def evaluate_survival_kfold(X: np.ndarray, df_surv: pd.DataFrame, k: int = 5, random_state: int = 0):
    """
    Patient-level GroupKFold not needed after pooling, but we keep kfold.
    Returns summary metrics + per-fold details.
    """
    # Build structured survival array
    y = Surv.from_arrays(event=df_surv["event"].astype(bool).values,
                         time=df_surv["time"].astype(float).values)

    # times for AUC evaluation: choose quantiles of observed times
    times = np.quantile(df_surv["time"].values, [0.25, 0.5, 0.75]).astype(float)
    times = np.unique(times)
    times = times[times > 0]

    # simple KFold over patients (since already pooled)
    # use GroupKFold over patient_id anyway for safety
    groups = df_surv["patient_id"].values
    cv = GroupKFold(n_splits=k)

    fold_rows = []
    cidxs = []
    auc_means = []

    model = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("cox", CoxPHSurvivalAnalysis(alpha=1e-4)),
    ])

    for fold, (tr, te) in enumerate(tqdm(cv.split(X, y, groups=groups), total=k, desc="CV folds"), start=1):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        model.fit(Xtr, ytr)

        # risk scores: higher => higher risk (shorter survival)
        risk_te = model.predict(Xte)

        # C-index
        c = concordance_index_censored(yte["event"], yte["time"], risk_te)[0]
        cidxs.append(float(c))

        # time-dependent AUC (cumulative dynamic)
        # needs training data to estimate IPCW
        auc, mean_auc = cumulative_dynamic_auc(ytr, yte, risk_te, times)
        auc_means.append(float(mean_auc))

        fold_rows.append({
            "fold": fold,
            "n_train": len(tr),
            "n_test": len(te),
            "c_index": float(c),
            "mean_auc": float(mean_auc),
            "times": times.tolist(),
            "auc_at_times": auc.tolist(),
        })

    out = pd.DataFrame(fold_rows)
    summary = {
        "k": k,
        "n_patients": len(df_surv),
        "c_index_mean": float(np.mean(cidxs)),
        "c_index_std": float(np.std(cidxs)),
        "mean_auc_mean": float(np.mean(auc_means)),
        "mean_auc_std": float(np.std(auc_means)),
        "times": times.tolist(),
    }
    return summary, out

## run for all 4 embedding sets + collect results

In [17]:
k = 5  # good default here

summaries = []
fold_tables = {}

for run_name, run_dir in RUNS.items():
    print("\n==============================")
    print("RUN:", run_name)
    print("DIR:", run_dir)

    E, names = load_run_embeddings(run_dir)
    emb_meta = build_embedding_meta(names)

    merged = attach_clinical(emb_meta, clinical)

    # Align embedding matrix rows with merged meta rows
    # emb_meta row order == E row order by construction
    # merged preserves that order, so we can index E by merged.index
    # but after merge, index stays original; make sure it matches:
    merged = merged.reset_index(drop=True)

    # pool to patient-level
    X_pat, df_pat = patient_pooling(E, merged)

    print("Patient-level X:", X_pat.shape, "patients:", df_pat["patient_id"].nunique())

    summary, folds = evaluate_survival_kfold(X_pat, df_pat, k=k)
    summary["run"] = run_name
    summaries.append(summary)
    fold_tables[run_name] = folds

results = pd.DataFrame(summaries).sort_values("c_index_mean", ascending=False)
results


RUN: MAE200_DINOe99
DIR: /proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch200/extracted_embeddings_epoch100
Matched via core_stem ↔ image_stem: 542/542
Patient-level X: (298, 768) patients: 298


CV folds: 100%|██████████| 5/5 [01:25<00:00, 17.13s/it]



RUN: MAE300_DINOe99
DIR: /proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch300/extracted_embeddings_epoch100
Matched via core_stem ↔ image_stem: 542/542
Patient-level X: (298, 768) patients: 298


CV folds: 100%|██████████| 5/5 [02:18<00:00, 27.80s/it]



RUN: MAE400_DINOe99
DIR: /proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch400/extracted_embeddings_epoch100
Matched via core_stem ↔ image_stem: 542/542
Patient-level X: (298, 768) patients: 298


CV folds: 100%|██████████| 5/5 [01:46<00:00, 21.38s/it]



RUN: MAE500_DINOe99
DIR: /proj/berzelius-2025-315/users/x_dilwi/SSL_method/results/DINO_from_MAE_epoch500/extracted_embeddings_epoch100
Matched via core_stem ↔ image_stem: 542/542
Patient-level X: (298, 768) patients: 298


CV folds: 100%|██████████| 5/5 [01:49<00:00, 21.85s/it]


,k,n_patients,c_index_mean,c_index_std,mean_auc_mean,mean_auc_std,times,run
0,5,298,0.522259,0.026572,0.546283,0.042201,"[660.5, 1729.5, 3342.75]",MAE200_DINOe99
3,5,298,0.510782,0.039255,0.499770,0.041932,"[660.5, 1729.5, 3342.75]",MAE500_DINOe99
1,5,298,0.503471,0.052989,0.495356,0.079670,"[660.5, 1729.5, 3342.75]",MAE300_DINOe99
2,5,298,0.493230,0.045975,0.485706,0.056440,"[660.5, 1729.5, 3342.75]",MAE400_DINOe99


In [18]:
best_run = results.iloc[0]["run"]
fold_tables[best_run]

,fold,n_train,n_test,c_index,mean_auc,times,auc_at_times
0,1,238,60,0.486453,0.532061,"[660.5, 1729.5, 3342.75]","[0.5244444444444445, 0.5140291806958474, 0.578..."
1,2,238,60,0.555348,0.594595,"[660.5, 1729.5, 3342.75]","[0.6354300385109114, 0.5792410714285715, 0.532..."
2,3,238,60,0.537805,0.555335,"[660.5, 1729.5, 3342.75]","[0.53125, 0.5305429864253394, 0.6532617820025]"
3,4,239,59,0.495478,0.472965,"[660.5, 1729.5, 3342.75]","[0.45454545454545453, 0.5045977011494253, 0.45..."
4,5,239,59,0.536210,0.576461,"[660.5, 1729.5, 3342.75]","[0.6428571428571429, 0.6040100250626567, 0.442..."
